In [2]:
from eyened_orm import (Database,
    ImageInstance,
    Feature, Segmentation, Modality,
    Series, Study, Patient, Project, Modality)
from eyened_orm import Creator
database = Database('/home/kvangarderen/.env')

In [ ]:
from eyened_orm import image_instance
import matplotlib.pyplot as plt



featurenames = ['RP featureset']
graders = ['Joline']
projects = ['OZR RPE65 Joline']

dataset_path = '/mnt/oogergo/eyened/nnunet/datasets/Dataset806_rpe65_featureset_v2/'
viz_path = '/mnt/ssd1/karin/test_nnunet_rp_featureset/viz'


with database.get_session() as session:
    instance_dict = {}
    patient_map = {}
    for featurename in featurenames:
        segmentation_list = session.query(Segmentation, ImageInstance, Patient.PatientIdentifier).join(ImageInstance)\
            .filter(ImageInstance.Modality == Modality.OCT)\
            .filter(Segmentation.Inactive == False)\
            .join(Creator).filter(Creator.CreatorName.in_(graders))\
            .join(Feature).filter(Feature.FeatureName == featurename)\
            .join(Series).join(Study).join(Patient).join(Project).filter(Project.ProjectName.in_(projects))\
            .all()
        
        print(featurename, len(segmentation_list))
        for seg, img, pid in segmentation_list:
            if img.ImageInstanceID not in instance_dict:
                instance_dict[img.ImageInstanceID] = {}
                patient_map[img.ImageInstanceID] = pid
            
            for bscan in seg.ScanIndices:
                if bscan not in instance_dict[img.ImageInstanceID]:
                    instance_dict[img.ImageInstanceID][bscan] = {n:None for n in featurenames}
                instance_dict[img.ImageInstanceID][bscan][featurename] = seg.SegmentationID
    
    print(instance_dict)


RP featureset 15
{2214559: {19: {'RP featureset': 43165}, 13: {'RP featureset': 43165}, 5: {'RP featureset': 43165}, 11: {'RP featureset': 43165}, 27: {'RP featureset': 43165}}, 2215105: {35: {'RP featureset': 43173}, 36: {'RP featureset': 43173}, 31: {'RP featureset': 43173}, 29: {'RP featureset': 43173}, 27: {'RP featureset': 43173}, 26: {'RP featureset': 43173}, 25: {'RP featureset': 43173}, 24: {'RP featureset': 43173}, 23: {'RP featureset': 43173}, 22: {'RP featureset': 43173}, 21: {'RP featureset': 43173}, 20: {'RP featureset': 43173}, 19: {'RP featureset': 43173}, 18: {'RP featureset': 43173}, 17: {'RP featureset': 43173}, 16: {'RP featureset': 43173}, 15: {'RP featureset': 43173}, 14: {'RP featureset': 43173}, 13: {'RP featureset': 43173}, 12: {'RP featureset': 43173}, 11: {'RP featureset': 43173}, 10: {'RP featureset': 43173}, 9: {'RP featureset': 43173}, 8: {'RP featureset': 43173}, 7: {'RP featureset': 43173}, 6: {'RP featureset': 43173}}, 2216143: {19: {'RP featureset': 431

In [ ]:

import numpy as np
import os
from PIL import Image
with database.get_session() as session:
    os.makedirs(viz_path, exist_ok=True)
    os.makedirs(dataset_path, exist_ok=True)
    os.makedirs(os.path.join(dataset_path, 'imagesTr'), exist_ok=True)
    os.makedirs(os.path.join(dataset_path, 'labelsTr'), exist_ok=True)
    for instanceid, obj in instance_dict.items():
        imdata = ImageInstance.by_id(session, instanceid).pixel_array
        
        for bscan, features in obj.items():
            bscandata = imdata[bscan]
            segmentation = np.zeros_like(bscandata)
            for i, featurename in enumerate(featurenames):
                segmentationid = features[featurename]
                if segmentationid is not None:
                    seg = Segmentation.by_id(session,segmentationid)
                    segdata = seg.read_data()
                    segmentation = segdata[bscan]
            plt.imshow(bscandata, cmap='gray')
            plt.imshow(segmentation, alpha=0.5*(segmentation>0))
            plt.axis('off')
            plt.savefig(os.path.join(viz_path, f'{instanceid}_{bscan}.png'), bbox_inches='tight', pad_inches=0)
            plt.close()
            imagepath = os.path.join(dataset_path, 'imagesTr', f'{instanceid}_{bscan}_0000.png')
            segpath = os.path.join(dataset_path, 'labelsTr', f'{instanceid}_{bscan}.png')
            Image.fromarray(bscandata).save(imagepath)
            Image.fromarray(segmentation).save(segpath)


    


## Create splits
Stratified by patient and balanced in terms of features

In [ ]:

import numpy as np

from sklearn.model_selection import GroupKFold
import numpy as np

kfold = GroupKFold(n_splits=5)
groups = []
sample_ids = []


for filename in os.listdir(os.path.join(dataset_path, 'imagesTr')):
    instanceid = filename.split('_')[0]
    patientid = patient_map[int(instanceid)]
    groups.append(patientid)
    sample_ids.append(filename.split('_0000')[0])

groups = np.array(groups)
sample_ids = np.array(sample_ids)

folds = list(kfold.split(np.zeros(len(groups)), groups=groups))

for i, (train_idx, val_idx) in enumerate(folds):

    patients = set(groups[val_idx])
    print(f"Fold {i}:")
    print(f"  Patients in validation set: {patients}")
    print(f"  Number of images in validation set: {len(val_idx)}")



## Write data to file

In [ ]:
import json


with database.get_session() as session:
    feature_featureset = session.query(Feature).filter(Feature.FeatureName == 'RP featureset').first()
    subfeatures = {f.Child.FeatureName: f.FeatureIndex for f in feature_featureset.FeatureAssociations}

dataset_json = {"channel_names": {"0": "OCT"}, 
    "labels": {"background": 0}, 
    "numTraining": 0, 
    "file_ending": ".png"}
for name, index in subfeatures.items():
    dataset_json["labels"][name] = index
json.dump(dataset_json, open(os.path.join(dataset_path, "dataset.json"), "w"))

splits_json = [{'train': [], 'val': []} for _ in range(len(folds))]
for i, (train_idx, val_idx) in enumerate(folds):
    splits_json[i]['val'] = list(sample_ids[val_idx])
    splits_json[i]['train'] = list(sample_ids[train_idx])
json.dump(splits_json, open(os.path.join(dataset_path, "splits_final.json"), "w"))



In [38]:
for i in range(5):
    print("Split %d: %d train, %d val" % (i, len(splits_json[i]['train']), len(splits_json[i]['val'])))

Split 0: 321 train, 168 val
Split 1: 345 train, 144 val
Split 2: 373 train, 116 val
Split 3: 458 train, 31 val
Split 4: 459 train, 30 val
